# 🏆 Kaggle Dataset Rankings EDA — Who Becomes a Grandmaster?
## 1,400 Users · 79 Grandmasters · 159 Masters · March 2026

**Dataset:** Kaggle Dataset Rankings — Experts, Masters & Grandmasters  
**Author:** Sergey Nefedov | [github.com/Sergpreneur](https://github.com/Sergpreneur)

---

### What this notebook covers
1. 📊 Leaderboard overview — tier distribution, points, medal counts
2. 🥇 Medal strategy analysis — gold vs volume vs balanced
3. 📈 Path to Grandmaster — what separates GMs from Experts?
4. ⏱️ Account age analysis — does experience matter?
5. 🔗 Correlations — what actually drives leaderboard points?
6. 🤖 Tier classification — can we predict who becomes a Grandmaster?

> **Key insight:** Reaching Grandmaster is not about volume — it's about gold medals.  
> The top 79 GMs have a median of 9 gold medals. The other 1,321 users have a median of 0.


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130, 'axes.facecolor': '#0d1117', 'figure.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e', 'text.color': '#c9d1d9',
    'grid.color': '#21262d', 'grid.alpha': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
})
BLUE='#388bfd'; GREEN='#3fb950'; RED='#f85149'; AMBER='#f7931a'
PURPLE='#9945ff'; TEAL='#39d353'; GRAY='#8b949e'
GOLD='#e8a020'; SILVER='#a0a0a0'; BRONZE='#b87333'

TIER_COLORS = {'Grandmaster': GOLD, 'Master': SILVER, 'Expert': BLUE}

PATH = '/kaggle/input/datasets/sergionefedov/kaggle-dataset-rankings-masters-grandmasters/'

df = pd.read_csv(PATH + 'kaggle_dataset_rankings.csv')

gm = df[df['tier']=='Grandmaster']
ma = df[df['tier']=='Master']
ex = df[df['tier']=='Expert']

print(f"Total users:    {len(df):,}")
print(f"Grandmasters:   {len(gm):,}  (rank 1–{gm['rank'].max()})")
print(f"Masters:        {len(ma):,}")
print(f"Experts:        {len(ex):,}")
print(f"\nPoints range:  {df['points'].min()} – {df['points'].max()}")
print(f"Max gold medals: {df['gold_medals'].max()} ({df.loc[df['gold_medals'].idxmax(),'username']})")
print(f"Max total medals: {df['total_medals'].max()} ({df.loc[df['total_medals'].idxmax(),'username']})")
print(f"\nMedal stats by tier:")
print(df.groupby('tier')[['gold_medals','silver_medals','bronze_medals','total_medals','points']].median().round(1).to_string())


---
## 1. 📊 Leaderboard Overview

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: Tier distribution donut
ax = axes[0,0]
tier_counts = df['tier'].value_counts().reindex(['Grandmaster','Master','Expert'])
tier_colors = [GOLD, SILVER, BLUE]
wedges,texts,autotexts = ax.pie(tier_counts.values, colors=tier_colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='#0d1117', linewidth=2),
    textprops={'fontsize':9,'color':'#c9d1d9'})
for at in autotexts: at.set_color('#c9d1d9')
ax.legend([f'{t} ({c:,})' for t,c in zip(tier_counts.index, tier_counts.values)],
          fontsize=8, loc='lower center')
ax.set_title('Tier Distribution', fontsize=11)
ax.text(0,0,f'{len(df):,}\nusers', ha='center', va='center', fontsize=12, color='#c9d1d9')

# Panel 2: Points distribution by tier
ax = axes[0,1]
for tier, color in TIER_COLORS.items():
    sub = df[df['tier']==tier]['points'].clip(0, 500)
    ax.hist(sub, bins=40, alpha=0.6, color=color, label=tier, density=True)
ax.set_title('Points Distribution by Tier', fontsize=11)
ax.set_xlabel('Points (capped at 500)'); ax.set_ylabel('Density')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 3: Total medals by tier (box)
ax = axes[0,2]
tier_order = ['Expert','Master','Grandmaster']
data_box = [df[df['tier']==t]['total_medals'].clip(0,80).values for t in tier_order]
bp = ax.boxplot(data_box, labels=tier_order, patch_artist=True, showfliers=False,
                medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], [BLUE, SILVER, GOLD]):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.set_title('Total Medals by Tier', fontsize=11)
ax.set_ylabel('Total Medals'); ax.grid(True, alpha=0.3, axis='y')

# Panel 4: Gold medals distribution (log scale)
ax = axes[1,0]
for tier, color in [('Grandmaster',GOLD),('Master',SILVER),('Expert',BLUE)]:
    sub = df[df['tier']==tier]['gold_medals']
    sub_nz = sub[sub > 0]
    if len(sub_nz):
        ax.hist(sub_nz, bins=25, alpha=0.6, color=color, label=f'{tier} (μ={sub.mean():.1f})', density=True)
ax.set_title('Gold Medal Distribution (holders only)', fontsize=11)
ax.set_xlabel('Gold Medals'); ax.set_ylabel('Density')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 5: Top 20 by points
ax = axes[1,1]
top20 = df.nlargest(20,'points')
colors_top = [TIER_COLORS[t] for t in top20['tier']]
ax.barh(range(20), top20['points'].values, color=colors_top, alpha=0.85)
ax.set_yticks(range(20))
ax.set_yticklabels([n[:20] for n in top20['username'].values], fontsize=7)
ax.set_title('Top 20 Users by Points', fontsize=11)
ax.set_xlabel('Points'); ax.grid(True, alpha=0.3, axis='x')
legend_patches = [mpatches.Patch(color=GOLD,label='Grandmaster'),
                  mpatches.Patch(color=SILVER,label='Master'),
                  mpatches.Patch(color=BLUE,label='Expert')]
ax.legend(handles=legend_patches, fontsize=7, loc='lower right')

# Panel 6: Account age distribution
ax = axes[1,2]
for tier, color in TIER_COLORS.items():
    sub = df[df['tier']==tier]['account_age_years'].dropna()
    ax.hist(sub, bins=20, alpha=0.5, color=color, label=f'{tier} (μ={sub.mean():.1f}y)', density=True)
ax.set_title('Account Age Distribution by Tier (years)', fontsize=11)
ax.set_xlabel('Account Age (years)'); ax.set_ylabel('Density')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Kaggle Dataset Rankings Overview', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('overview.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("\nMedian account age by tier:")
print(df.groupby('tier')['account_age_years'].median().round(2).to_string())


---
## 2. 🥇 Medal Strategy Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel 1: Strategy distribution
ax = axes[0,0]
strat_counts = df['strategy'].value_counts()
strat_colors_map = {'gold_focused':GOLD,'balanced':GREEN,'volume':AMBER,'points_only':GRAY}
sc = [strat_colors_map.get(s,GRAY) for s in strat_counts.index]
wedges,texts,autotexts = ax.pie(strat_counts.values, colors=sc,
    autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(edgecolor='#0d1117',linewidth=1.5),
    textprops={'fontsize':9,'color':'#c9d1d9'})
for at in autotexts: at.set_color('#c9d1d9')
ax.legend([f'{s} ({c})' for s,c in zip(strat_counts.index, strat_counts.values)], fontsize=8)
ax.set_title('Strategy Distribution', fontsize=11)

# Panel 2: Strategy by tier heatmap
ax = axes[0,1]
strat_tier = df.groupby(['tier','strategy']).size().unstack(fill_value=0)
strat_tier_pct = strat_tier.div(strat_tier.sum(axis=1), axis=0)*100
strat_tier_pct = strat_tier_pct.reindex(['Expert','Master','Grandmaster'])
sns.heatmap(strat_tier_pct, annot=True, fmt='.0f', cmap='YlOrRd',
            ax=ax, linewidths=0.3, cbar_kws={'label':'% of tier'},
            annot_kws={'size':10})
ax.set_title('Strategy Distribution by Tier (%)', fontsize=11)
ax.set_xlabel('Strategy'); ax.set_ylabel('')

# Panel 3: Points by strategy
ax = axes[0,2]
for strat, color in strat_colors_map.items():
    sub = df[df['strategy']==strat]['points'].clip(0,400)
    if len(sub) > 5:
        ax.hist(sub, bins=30, alpha=0.6, color=color,
                label=f'{strat} (med={sub.median():.0f})', density=True)
ax.set_title('Points Distribution by Strategy', fontsize=11)
ax.set_xlabel('Points (capped 400)'); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)

# Panel 4: Medal composition treemap-style bar
ax = axes[1,0]
strats = ['gold_focused','balanced','volume']
gold_m  = [df[df['strategy']==s]['gold_medals'].mean() for s in strats]
silver_m= [df[df['strategy']==s]['silver_medals'].mean() for s in strats]
bronze_m= [df[df['strategy']==s]['bronze_medals'].mean() for s in strats]
x_ = np.arange(len(strats))
w_ = 0.6
ax.bar(x_, gold_m,   w_, color=GOLD,   alpha=0.85, label='Gold')
ax.bar(x_, silver_m, w_, color=SILVER, alpha=0.85, label='Silver', bottom=gold_m)
ax.bar(x_, bronze_m, w_, color=BRONZE, alpha=0.85, label='Bronze',
       bottom=[g+s for g,s in zip(gold_m,silver_m)])
ax.set_xticks(x_); ax.set_xticklabels(strats, fontsize=9)
ax.set_title('Mean Medal Composition by Strategy', fontsize=11)
ax.set_ylabel('Mean Medal Count'); ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Panel 5: Gold rate % by tier
ax = axes[1,1]
tier_gold_rate = df.groupby('tier')['gold_rate_pct'].median().reindex(['Expert','Master','Grandmaster'])
ax.bar(range(3), tier_gold_rate.values, color=[BLUE,SILVER,GOLD], alpha=0.85)
ax.set_xticks(range(3)); ax.set_xticklabels(['Expert','Master','Grandmaster'])
ax.set_title('Median Gold Rate % by Tier', fontsize=11)
ax.set_ylabel('Gold Rate (% of total medals)')
ax.grid(True, alpha=0.3, axis='y')
for i,v in enumerate(tier_gold_rate.values):
    ax.text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Panel 6: Medals per year by tier
ax = axes[1,2]
for tier, color in [('Expert',BLUE),('Master',SILVER),('Grandmaster',GOLD)]:
    sub = df[df['tier']==tier]['medals_per_year'].clip(0,30)
    ax.hist(sub, bins=25, alpha=0.6, color=color,
            label=f'{tier} (med={sub.median():.1f})', density=True)
ax.set_title('Medals per Year by Tier', fontsize=11)
ax.set_xlabel('Medals / Year'); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

plt.suptitle('Medal Strategy Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('strategy.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Strategy × tier cross-tab
print("\nStrategy distribution within each tier (%):")
print(strat_tier_pct.round(1).to_string())


---
## 3. 📈 Path to Grandmaster — What Separates GMs?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel 1: Gold medals — GM vs non-GM
ax = axes[0,0]
non_gm = df[df['tier']!='Grandmaster']['gold_medals'].clip(0,20)
gm_gold = gm['gold_medals'].clip(0,60)
ax.hist(non_gm, bins=20, alpha=0.65, color=BLUE, label=f'Expert+Master (med={non_gm.median():.0f})', density=True)
ax.hist(gm_gold, bins=20, alpha=0.7, color=GOLD, label=f'Grandmaster (med={gm_gold.median():.0f})', density=True)
ax.axvline(5, color=RED, linewidth=1.5, linestyle='--', label='GM threshold (5 gold)')
ax.set_title('Gold Medals: Grandmasters vs Others', fontsize=11)
ax.set_xlabel('Gold Medals'); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

# Panel 2: Silver medals — same comparison
ax = axes[0,1]
non_gm_s = df[df['tier']!='Grandmaster']['silver_medals'].clip(0,30)
gm_s = gm['silver_medals'].clip(0,100)
ax.hist(non_gm_s, bins=25, alpha=0.65, color=BLUE, label=f'Expert+Master (med={non_gm_s.median():.0f})', density=True)
ax.hist(gm_s, bins=25, alpha=0.7, color=SILVER, label=f'Grandmaster (med={gm_s.median():.0f})', density=True)
ax.axvline(5, color=RED, linewidth=1.5, linestyle='--', label='GM threshold (5 silver)')
ax.set_title('Silver Medals: Grandmasters vs Others', fontsize=11)
ax.set_xlabel('Silver Medals'); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

# Panel 3: Scatter gold vs silver (all tiers)
ax = axes[0,2]
for tier, color, size in [('Expert',BLUE,8),('Master',SILVER,12),('Grandmaster',GOLD,18)]:
    sub = df[df['tier']==tier]
    ax.scatter(sub['gold_medals'].clip(0,55), sub['silver_medals'].clip(0,95),
               color=color, alpha=0.5, s=size, label=tier)
ax.axvline(5, color=RED, linewidth=1, linestyle='--', alpha=0.6)
ax.axhline(5, color=RED, linewidth=1, linestyle='--', alpha=0.6)
ax.set_title('Gold vs Silver Medals (GM quadrant highlighted)', fontsize=11)
ax.set_xlabel('Gold Medals'); ax.set_ylabel('Silver Medals')
ax.text(6, 85, 'GM zone', fontsize=9, color=GOLD)
ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

# Panel 4: Points needed per tier
ax = axes[1,0]
tier_points = df.groupby('tier')['points'].agg(['min','median','max']).reindex(['Expert','Master','Grandmaster'])
x_ = np.arange(3)
ax.bar(x_-0.2, tier_points['min'],    0.2, color=BLUE,  alpha=0.85, label='Min')
ax.bar(x_,     tier_points['median'], 0.2, color=GREEN, alpha=0.85, label='Median')
ax.bar(x_+0.2, tier_points['max'],    0.2, color=RED,   alpha=0.85, label='Max')
ax.set_xticks(x_); ax.set_xticklabels(['Expert','Master','Grandmaster'])
ax.set_yscale('log')
ax.set_title('Points Distribution by Tier (log scale)', fontsize=11)
ax.set_ylabel('Points (log)'); ax.legend(fontsize=9); ax.grid(True,alpha=0.3,axis='y')

# Panel 5: Minimum GM profile
ax = axes[1,1]
gm_sorted = gm.sort_values('rank')
ax.scatter(gm_sorted['gold_medals'], gm_sorted['silver_medals'],
           c=gm_sorted['rank'], cmap='YlOrRd_r', s=60, alpha=0.8)
ax.axvline(5, color=GRAY, linewidth=1, linestyle='--', alpha=0.5)
ax.axhline(5, color=GRAY, linewidth=1, linestyle='--', alpha=0.5)
last_gm = gm_sorted.iloc[-1]
ax.annotate(f"Last GM({last_gm['gold_medals']}G/{last_gm['silver_medals']}S)",(last_gm['gold_medals'], last_gm['silver_medals']),xytext=(last_gm['gold_medals']+2, last_gm['silver_medals']+3),fontsize=8, color=RED)
ax.set_title('All Grandmasters: Gold vs Silver (rank = color)', fontsize=11)
ax.set_xlabel('Gold Medals'); ax.set_ylabel('Silver Medals')
ax.grid(True,alpha=0.3)

# Panel 6: Rank vs total medals
ax = axes[1,2]
sample = df.sample(min(500,len(df)), random_state=42)
for tier, color in TIER_COLORS.items():
    sub = df[df['tier']==tier]
    ax.scatter(sub['total_medals'], sub['rank'],
               color=color, alpha=0.4, s=8, label=tier)
ax.set_title('Rank vs Total Medals', fontsize=11)
ax.set_xlabel('Total Medals'); ax.set_ylabel('Rank (lower = better)')
ax.invert_yaxis()
ax.legend(fontsize=8); ax.grid(True,alpha=0.3)
corr,_ = stats.spearmanr(df['total_medals'], df['rank'])
ax.text(0.05,0.08,f'Spearman r = {corr:.3f}',transform=ax.transAxes,
        fontsize=9,bbox=dict(boxstyle='round',facecolor='#21262d',alpha=0.8))

plt.suptitle('Path to Grandmaster Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('gm_path.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Statistical test: are GM gold medals significantly different?
stat, pval = stats.mannwhitneyu(gm['gold_medals'], ex['gold_medals'], alternative='greater')
print(f"Mann-Whitney U test: GM gold > Expert gold")
print(f"  statistic={stat:.0f}, p={pval:.2e} {'✅ significant' if pval<0.05 else '❌ not significant'}")
print(f"\nGM gold medals: median={gm['gold_medals'].median():.0f}, mean={gm['gold_medals'].mean():.1f}")
print(f"Expert gold medals: median={ex['gold_medals'].median():.0f}, mean={ex['gold_medals'].mean():.1f}")


---
## 4. ⏱️ Account Age & Experience Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Medals per year vs tier
ax = axes[0]
data_mpy = [df[df['tier']==t]['medals_per_year'].clip(0,20).values
            for t in ['Expert','Master','Grandmaster']]
bp = ax.boxplot(data_mpy, labels=['Expert','Master','Grandmaster'],
                patch_artist=True, showfliers=False,
                medianprops=dict(color='white',linewidth=2))
for patch,color in zip(bp['boxes'],[BLUE,SILVER,GOLD]):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.set_title('Medals per Year by Tier', fontsize=11)
ax.set_ylabel('Medals / Year'); ax.grid(True,alpha=0.3,axis='y')

# Panel 2: Account age vs gold medals (GMs only)
ax = axes[1]
sc = ax.scatter(gm['account_age_years'], gm['gold_medals'],
                c=gm['points'], cmap='YlOrRd', s=60, alpha=0.8)
plt.colorbar(sc, ax=ax, label='Points')
ax.set_title('GM: Account Age vs Gold Medals', fontsize=11)
ax.set_xlabel('Account Age (years)'); ax.set_ylabel('Gold Medals')
ax.grid(True,alpha=0.3)
corr_gm,_ = stats.spearmanr(gm['account_age_years'].dropna(), gm['gold_medals'])
ax.text(0.05,0.92,f'r = {corr_gm:.3f}',transform=ax.transAxes,fontsize=9,
        bbox=dict(boxstyle='round',facecolor='#21262d',alpha=0.8))

# Panel 3: New vs established — can newcomers reach top?
ax = axes[2]
df['is_new'] = df['account_age_years'] < 1
new_tier = df.groupby(['is_new','tier']).size().unstack(fill_value=0)
new_pct = new_tier.div(new_tier.sum(axis=1),axis=0)*100
labels_n = ['Established (1+ yr)','New (< 1 yr)']
x_ = np.arange(3); w_=0.38
tiers_n = ['Expert','Master','Grandmaster']
tc = [BLUE,SILVER,GOLD]
for i,(t,c) in enumerate(zip(tiers_n,tc)):
    vals = [new_pct.loc[False,t] if False in new_pct.index and t in new_pct.columns else 0,
            new_pct.loc[True,t]  if True  in new_pct.index and t in new_pct.columns else 0]
    ax.bar([0+i*0.28, 1+i*0.28], vals, 0.25, color=c, alpha=0.85, label=t)
ax.set_xticks([0.28,1.28]); ax.set_xticklabels(labels_n, fontsize=9)
ax.set_title('Tier % by Account Age Group', fontsize=11)
ax.set_ylabel('%'); ax.legend(fontsize=8); ax.grid(True,alpha=0.3,axis='y')

plt.suptitle('Account Age & Experience', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('age_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("Median account age:")
print(df.groupby('tier')['account_age_years'].agg(['median','mean']).round(2).to_string())
new_gms = (gm['account_age_years'] < 1).sum()
print(f"\nGrandmasters with account < 1 year: {new_gms} ({new_gms/len(gm):.1%})")


---
## 5. 🔗 Correlations — What Drives Points?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel 1: Correlation matrix
ax = axes[0]
num_cols = ['rank','gold_medals','silver_medals','bronze_medals','total_medals',
            'points','account_age_years','medals_per_year','gold_rate_pct','points_per_medal']
corr_matrix = df[num_cols].corr(method='spearman')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.3, cbar_kws={'label':'Spearman r'},
            annot_kws={'size':8})
ax.set_title('Spearman Correlation Matrix', fontsize=11)
plt.setp(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), fontsize=8)

# Panel 2: Feature importance for predicting points
ax = axes[1]
features = ['gold_medals','silver_medals','bronze_medals','total_medals',
            'account_age_years','medals_per_year','gold_rate_pct']
corrs_with_points = {}
for feat in features:
    sub = df[[feat,'points']].dropna()
    r,_ = stats.spearmanr(sub[feat], sub['points'])
    corrs_with_points[feat] = r

corr_series = pd.Series(corrs_with_points).sort_values(ascending=True)
ax.barh(corr_series.index, corr_series.values,
        color=[GREEN if v>0 else RED for v in corr_series.values], alpha=0.85)
ax.axvline(0, color=GRAY, linewidth=0.8)
ax.set_title('Spearman Correlation with Points', fontsize=11)
ax.set_xlabel('Spearman r'); ax.grid(True,alpha=0.3,axis='x')
for i,v in enumerate(corr_series.values):
    ax.text(v+0.01 if v>=0 else v-0.01, i, f'{v:.3f}',
            va='center', ha='left' if v>=0 else 'right', fontsize=9)

plt.suptitle('What Drives Leaderboard Points?', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('correlations.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("Top correlations with points:")
print(corr_series.sort_values(ascending=False).round(3).to_string())


---
## 6. 🤖 Tier Classification — Can We Predict Grandmaster?

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

FEATURES = ['gold_medals','silver_medals','bronze_medals','total_medals',
            'account_age_years','medals_per_year','gold_rate_pct',
            'gold_to_silver_ratio','points_per_medal']

X = df[FEATURES].fillna(0)
y = df['tier'].map({'Expert':0,'Master':1,'Grandmaster':2})

# Binary: GM vs non-GM (more useful)
y_binary = (df['tier']=='Grandmaster').astype(int)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gbm   = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
lr    = Pipeline([('sc',RobustScaler()),('clf',LogisticRegression(C=1.0,max_iter=500,class_weight='balanced'))])

gbm_scores  = cross_val_score(gbm,  X, y_binary, cv=skf, scoring='roc_auc')
lr_scores   = cross_val_score(lr,   X, y_binary, cv=skf, scoring='roc_auc')

gbm.fit(X, y_binary)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Feature importance
ax = axes[0]
fi = pd.Series(gbm.feature_importances_, index=FEATURES).sort_values(ascending=True)
ax.barh(fi.index, fi.values,
        color=[GOLD if 'gold' in f else SILVER if 'silver' in f else BLUE for f in fi.index],
        alpha=0.85)
ax.set_title('Feature Importance (GBM — GM vs non-GM)', fontsize=11)
ax.set_xlabel('Importance'); ax.grid(True,alpha=0.3,axis='x')

# Panel 2: CV scores comparison
ax = axes[1]
models = ['Logistic Reg','Gradient Boosting']
scores = [lr_scores, gbm_scores]
for i,(name,sc) in enumerate(zip(models,scores)):
    ax.bar(i, sc.mean(), color=[BLUE,GREEN][i], alpha=0.8, width=0.4)
    ax.errorbar(i, sc.mean(), yerr=sc.std()*2, color='white', linewidth=2, capsize=8)
    ax.text(i, sc.mean()+0.01, f'{sc.mean():.3f}±{sc.std():.3f}',
            ha='center', fontsize=9)
ax.set_xticks([0,1]); ax.set_xticklabels(models)
ax.set_ylim(0.8, 1.02)
ax.set_title('ROC-AUC (5-fold CV) — Grandmaster Classification', fontsize=11)
ax.set_ylabel('ROC-AUC'); ax.grid(True,alpha=0.3,axis='y')
ax.axhline(0.5, color=GRAY, linewidth=0.8, linestyle='--', label='Random')
ax.legend(fontsize=8)

# Panel 3: Confusion matrix (GBM, full fit)
ax = axes[2]
from sklearn.model_selection import cross_val_predict
y_pred = cross_val_predict(gbm, X, y_binary, cv=skf)
cm = confusion_matrix(y_binary, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not GM','Grandmaster'],
            yticklabels=['Not GM','Grandmaster'],
            annot_kws={'size':12})
ax.set_title('Confusion Matrix (5-fold CV)', fontsize=11)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.suptitle(f'Tier Classification | GBM AUC={gbm_scores.mean():.3f} | LR AUC={lr_scores.mean():.3f}',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('classification.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"GBM ROC-AUC: {gbm_scores.mean():.4f} ± {gbm_scores.std():.4f}")
print(f"LR  ROC-AUC: {lr_scores.mean():.4f} ± {lr_scores.std():.4f}")
print(f"\nGBM top feature: {fi.idxmax()} ({fi.max():.3f})")
print(f"\nClassification report (GBM CV predictions):")
print(classification_report(y_binary, y_pred, target_names=['Not GM','Grandmaster']))


---
## 7. 📋 Key Findings

**Tier distribution is sharply skewed:**
Grandmaster (5.6%) → Master (11.4%) → Expert (83%) — classic power law. Most ranked users are Experts who have just cleared the minimum 3-bronze threshold.

**Gold medals are the decisive separator:**
Median GM has 9 gold medals. Median Expert has 0. Gold medal count alone achieves ~95%+ ROC-AUC in Grandmaster classification — it's the single most predictive feature by far.

**Two winning strategies at the top:**
Rank #1 (Umer Haddii): 51 gold, 0 bronze — pure quality focus, 76% gold rate.
Rank #2 (Kanchana1990): 24 gold + 90 silver + 46 bronze — maximum volume, 160 total medals.
Both work, but the gold-focused path scores higher points per medal.

**Account age matters less than expected:**
Several GMs have accounts under 2 years old — the path to GM is about output quality, not time served. However, GMs have higher medals/year rates, suggesting they publish more consistently.

**Points correlate strongly with gold medals (r ≈ 0.85+):**
Silver and bronze contribute, but gold medals dominate the points calculation. Optimising for gold upvotes (25+) is the most efficient path to leaderboard rank.

---

*Dataset & notebook by **Sergey Nefedov** | [github.com/Sergpreneur](https://github.com/Sergpreneur)*  
*If this analysis was useful, an upvote is greatly appreciated! 🙏*
